# Enterprise Events And Observability Graphs

This notebook covers enterprise loaders and mappers that are easy to overlook: web pages, HTML metadata, logs, event payloads, trace/span correlation, scientific papers, and JSON-LD loader output.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "src").exists():
            ROOT = parent
            break
sys.path.insert(0, str(ROOT / "src"))


## Load web, HTML, log, event, scientific, and JSON-LD sources

All loaders return `SourceRecord` objects with stable source IDs and source-level evidence.


In [ ]:
from pathlib import Path
import json
import tempfile

from kg_construction.infrastructure.loaders import (
    EventPayloadLoader,
    HTMLLoader,
    JSONLDLoader,
    LogLoader,
    ScientificPaperLoader,
    WebPageLoader,
)
from kg_construction.infrastructure.mapping import EventGraphMapper, HTMLMetadataMapper, LogEventMapper

FIXTURES = ROOT / "tests" / "fixtures" / "enterprise_loaders"

html_source = HTMLLoader().load(FIXTURES / "recommendation_runbook.html")[0]
webpage_source = WebPageLoader().load(FIXTURES / "recommendation_runbook.html")[0]
log_source = LogLoader().load(FIXTURES / "service_events.log")[0]
paper_source = ScientificPaperLoader().load(FIXTURES / "kg_construction_article.jats")[0]

with tempfile.TemporaryDirectory() as tmpdir:
    jsonld_path = Path(tmpdir) / "service.jsonld"
    jsonld_path.write_text(
        json.dumps({"@context": {"name": "http://schema.org/name"}, "@id": "urn:svc:recommendation", "name": "Recommendation API"}),
        encoding="utf-8",
    )
    jsonld_source = JSONLDLoader().load(jsonld_path)[0]

{
    "html": (html_source.source_type, html_source.metadata["title"]),
    "webpage": (webpage_source.source_type, webpage_source.metadata["loader"]),
    "logs": (log_source.source_type, log_source.metadata["row_count"]),
    "paper": (paper_source.source_type, paper_source.metadata["doi"]),
    "jsonld": (jsonld_source.source_type, bool(jsonld_source.metadata.get("source_evidence"))),
}


## HTML metadata mapper

The mapper turns document metadata, links, and embedded JSON-LD/schema.org blocks into graph candidates.


In [ ]:
html_batch = HTMLMetadataMapper().map_source(html_source)
{
    "entities": sorted({entity.entity_type for entity in html_batch.entities}),
    "relations": sorted({relation.relation_type for relation in html_batch.relations}),
    "json_ld_count": html_batch.metadata["json_ld_count"],
}


## Log template mining and log event graph mapping

The log loader normalizes Apache, JSON, key-value, and generic logs. The mapper creates log events, services, endpoints, levels, and template nodes.


In [ ]:
log_batch = LogEventMapper().map_source(log_source)
{
    "templates": sorted({row.get("template") for row in log_source.records if row.get("template")}),
    "entity_types": sorted({entity.entity_type for entity in log_batch.entities}),
    "relation_types": sorted({relation.relation_type for relation in log_batch.relations}),
}


## Event payload graph mapping with traces, spans, and causality

Event payloads can produce event, service, trace, span, parent-span, and caused-by relationships.


In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    event_path = Path(tmpdir) / "events.json"
    event_path.write_text(
        json.dumps(
            {
                "stream": "service-events",
                "events": [
                    {"event_id": "e1", "event_type": "request", "service": "Recommendation API", "trace_id": "t1", "span_id": "s1"},
                    {"event_id": "e2", "event_type": "db", "service": "Recommendation API", "trace_id": "t1", "span_id": "s2", "parent_span_id": "s1", "caused_by": "e1"},
                ],
            }
        ),
        encoding="utf-8",
    )
    event_source = EventPayloadLoader().load(event_path)[0]
    event_batch = EventGraphMapper().map_source(event_source)

{
    "event_count": event_batch.metadata["event_count"],
    "entity_types": sorted({entity.entity_type for entity in event_batch.entities}),
    "relation_types": sorted({relation.relation_type for relation in event_batch.relations}),
    "trace_span_correlation": event_batch.metadata["trace_span_correlation"],
}
